# Round 2 experiment: LFA at earlier layers + higher resolution

**What this does**: the exact same 20 folds as last time (reusing `runs_phase0/folds.csv` and the cached resized images),
the same training procedure, the same two seeds. Only two things change:

| Config | LFA position | Input resolution |
|---|---|---|
| `none_224` | None (baseline, rerun) | 224 |
| `lfa14_224` | After the 5th stage (14×14, 112 channels) | 224 |
| `lfa28_224` | After the 3rd stage (28×28, 40 channels) | 224 |
| `none_320` | None | 320 |
| `lfa14_320` | After the 5th stage (20×20, 112 channels) | 320 |

5 configs × 20 folds × 2 seeds = 200 runs. Each 224 run takes about 1.5 minutes and each 320 run about 3 minutes,
**about 7 hours in total**. Run the three 224 configs first (about 3 hours), then the two 320 configs.

**Decision rule, fixed in advance** (section 3 writes it to `PREREG.md` on Drive together with the time, before any run starts):
The 4 primary comparisons are all paired against `none_224`, using the corrected resampled t-test with Holm correction for multiple comparisons.
It counts as an improvement if the mean gain is ≥ 2 points and the Holm-adjusted p < 0.05. Results are reported as they are, whatever they turn out to be.

Interruptions are fine: each run is written to Drive as soon as it finishes, and "Run all" again skips the completed ones.

Before running: "Runtime → Change runtime type" and choose **GPU** (A100 or L4 both work).
In the morning: run the last cell ("Analysis") and send the output to Claude.

## 1. Mount Drive, check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or '⚠ No GPU! Change the runtime type')

## 2. Settings

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/p1_reconstructure')
SRC_DIR   = DRIVE_DIR / 'runs_phase0'      # last time's cache_512.pkl, folds.csv and results.csv are all here
OUT_DIR   = DRIVE_DIR / 'runs_v2'          # results of this run are written here

# config name: (LFA inserted after features[k], None = no LFA; input resolution)
CONFIGS = {
    'none_224':  (None, 224),
    'lfa14_224': (5, 224),
    'lfa28_224': (3, 224),
    'none_320':  (None, 320),
    'lfa14_320': (5, 320),
}
PHASES = [['none_224', 'lfa14_224', 'lfa28_224'], ['none_320', 'lfa14_320']]
PRIMARY = ['lfa14_224', 'lfa28_224', 'none_320', 'lfa14_320']   # all compared against none_224
SEEDS = [0, 1]
EPOCHS_S1, EPOCHS_S2 = 15, 15
LR_S1, LR_S2 = 1e-3, 1e-5
BATCH = 16
WORKERS = 2
QUICK_TEST = False   # True = 1 epoch per stage and only 2 folds, a few-minute trial to check that it runs

## 3. Write down the rules fixed in advance (written only on the first run, never overwritten)

In [ ]:
import datetime
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREREG = OUT_DIR / 'PREREG.md'
if not PREREG.exists() and not QUICK_TEST:
    PREREG.write_text(f"""# Pre-specified analysis, written {datetime.datetime.now().isoformat(timespec='seconds')} before any v2 run

Folds: runs_phase0/folds.csv (20 grouped folds, as in the first experiment). Seeds 0 and 1.
Training protocol identical to the first experiment except for the two factors below.

Configurations: {CONFIGS}
LFA = 1x1 conv to one channel, sigmoid, element-wise multiplication (default PyTorch init).
features[3] output: 40 ch, stride 8; features[5] output: 112 ch, stride 16.
Resolution s: train RandomResizedCrop(s, scale 0.7-1.0); test Resize(round(s*256/224)) + CenterCrop(s).

Primary metric: macro F1 over classes present in each test fold; seeds averaged within fold.
Primary comparisons (each vs none_224): {PRIMARY}.
Test: Nadeau-Bengio corrected resampled t-test over 20 folds, factor 1/20 + mean n_test / mean (n_train + n_val).
Multiplicity: Holm over the 4 primary comparisons.
Counted as an improvement only if mean paired difference >= 2 pp AND Holm-adjusted p < 0.05.
Secondary (descriptive): lfa14_320 vs none_320; none_224 here vs 'none' in runs_phase0 (rerun agreement).
All results are reported whatever their direction.
""", encoding='utf-8')
print(PREREG.read_text(encoding='utf-8') if PREREG.exists() else '(quick-test mode, PREREG not written)')

## 4. Load the image cache and last time's 20 folds

In [ ]:
import io, re, pickle, time, copy, random
import numpy as np
import pandas as pd
from PIL import Image

CLASSES_ALL = ['Algal', 'Leaf_rot', 'Phomopsis', 'Pink_disease', 'Root_disease']


def norm_class(part):
    if part.startswith('Leaf_rot'):
        return 'Leaf_rot'
    for c in CLASSES_ALL:
        if part.lower() == c.lower():
            return c
    return None


with open(SRC_DIR / 'cache_512.pkl', 'rb') as f:
    cache = pickle.load(f)
assert len(cache['images']) == 560, len(cache['images'])
meta = pd.read_csv(io.StringIO(cache['sessions_csv']))
meta['cls'] = meta['cls'].map(norm_class)
meta = meta[meta['cls'] != 'Pink_disease'].reset_index(drop=True)
CLASSES = sorted(meta['cls'].unique(), key=CLASSES_ALL.index)
meta['y'] = meta['cls'].map({c: i for i, c in enumerate(CLASSES)})
folds = pd.read_csv(SRC_DIR / 'folds.csv')
assert len(meta) == 550 and folds.groupby(['repeat', 'fold']).size().eq(550).all()
print(f'{len(meta)} images, classes {CLASSES}, {folds.groupby(["repeat", "fold"]).ngroups} folds')

## 5. Model

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class LesionFocusAttention(nn.Module):
    """Identical to round 1: 1x1 conv to 1 channel → sigmoid → element-wise multiplication."""

    def __init__(self, in_channels):
        super().__init__()
        self.attention_conv = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.attention_conv(x))


def build_model(pos, n_cls, pretrained=True):
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
    for p in m.parameters():
        p.requires_grad = False
    lfa = None
    if pos is not None:
        blocks = list(m.features.children())
        ch = blocks[pos][-1].out_channels
        lfa = LesionFocusAttention(ch)
        m.features = nn.Sequential(*blocks[:pos + 1], lfa, *blocks[pos + 1:])
    m.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True), nn.Linear(m.classifier[1].in_features, n_cls))
    for p in m.classifier.parameters():
        p.requires_grad = True
    if lfa is not None:
        for p in lfa.parameters():
            p.requires_grad = True
    return m


for name, (pos, size) in CONFIGS.items():
    mm = build_model(pos, 4, pretrained=False)
    lfa_mods = [x for x in mm.modules() if isinstance(x, LesionFocusAttention)]
    with torch.no_grad():
        mm.eval()
        out = mm(torch.zeros(1, 3, size, size))
    extra = sum(p.numel() for x in lfa_mods for p in x.parameters())
    print(f'{name:<10} LFA params {extra:>4}  output {tuple(out.shape)}')

## 6. Training functions (same as round 1, only the resolution is variable)

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def make_tfs(size):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
        transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 1.5)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.3, scale=(0.02, 0.10)),
    ])
    test_tf = transforms.Compose([transforms.Resize(round(size * 256 / 224)), transforms.CenterCrop(size),
                                  transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return train_tf, test_tf


class ImageSet(Dataset):
    def __init__(self, ims, ys, tf):
        self.ims, self.ys, self.tf = list(ims), list(ys), tf

    def __len__(self):
        return len(self.ims)

    def __getitem__(self, i):
        return self.tf(self.ims[i]), int(self.ys[i])


def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _worker_init(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    out, ys = [], []
    for X, y in loader:
        with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            out.append(model(X.to(DEVICE)).float().cpu())
        ys += y.tolist()
    return torch.cat(out).numpy(), np.array(ys)


def macro_f1_present(yt, yp):
    return f1_score(yt, yp, labels=sorted(set(yt)), average='macro', zero_division=0)


def train_model(pos, size, ims_tr, y_tr, ims_va, y_va, n_cls, seed, epochs, pretrained=True):
    seed_everything(seed)
    train_tf, test_tf = make_tfs(size)
    y_tr = np.asarray(y_tr)
    counts = np.bincount(y_tr, minlength=n_cls)
    cw = torch.tensor(len(y_tr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
    sw = (1.0 / np.maximum(counts, 1))[y_tr]
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True, generator=g)
    kw = dict(num_workers=WORKERS, worker_init_fn=_worker_init, pin_memory=(DEVICE == 'cuda'))
    tl = DataLoader(ImageSet(ims_tr, y_tr, train_tf), batch_size=BATCH, sampler=sampler, generator=g, **kw)
    vl = DataLoader(ImageSet(ims_va, y_va, test_tf), batch_size=64, shuffle=False, **kw)
    model = build_model(pos, n_cls, pretrained).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    best = [-1.0, None]
    for n_ep, lr, unfreeze in [(epochs[0], LR_S1, False), (epochs[1], LR_S2, True)]:
        if unfreeze:
            model.load_state_dict(best[1])
            for p in model.parameters():
                p.requires_grad = True
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        for _ in range(n_ep):
            model.train()
            for X, y in tl:
                X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                    loss = crit(model(X), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            logits, yv = predict_logits(model, vl)
            f1v = macro_f1_present(yv, logits.argmax(1))
            if f1v > best[0]:
                best = [f1v, copy.deepcopy(model.state_dict())]
    model.load_state_dict(best[1])
    return model, best[0], test_tf


def to_pil(idx):
    return [Image.open(io.BytesIO(cache['images'][(meta.at[i, 'cls'], meta.at[i, 'file'])])).convert('RGB')
            for i in idx]

## 7. Main loop (can be interrupted and resumed)

In [ ]:
res_csv, pred_csv = OUT_DIR / ('results_quick.csv' if QUICK_TEST else 'results.csv'), \
                    OUT_DIR / ('preds_quick.csv' if QUICK_TEST else 'preds.csv')
done = set()
if res_csv.exists():
    r0 = pd.read_csv(res_csv)
    done = set(zip(r0.config, r0.repeat, r0.fold, r0.seed))
keys = sorted(folds.groupby(['repeat', 'fold']).groups)[: 2 if QUICK_TEST else None]
jobs = [(c, rep, k, s) for ph in PHASES for (rep, k) in keys for s in SEEDS for c in ph]
todo = [j for j in jobs if j not in done]
epochs = (1, 1) if QUICK_TEST else (EPOCHS_S1, EPOCHS_S2)
print(f'{len(jobs)} runs in total, {len(jobs) - len(todo)} done, {len(todo)} left')
t_start = time.time()
for n, (cfg, rep, k, s) in enumerate(todo, 1):
    pos, size = CONFIGS[cfg]
    f = folds[(folds.repeat == rep) & (folds.fold == k)]
    tr, va, te = (f[f.role == r].idx.values for r in ['train', 'val', 'test'])
    t0 = time.time()
    model, vf1, test_tf = train_model(pos, size, to_pil(tr), meta.y.values[tr], to_pil(va), meta.y.values[va],
                                      len(CLASSES), s, epochs)
    tl = DataLoader(ImageSet(to_pil(te), meta.y.values[te], test_tf), batch_size=64, shuffle=False, num_workers=WORKERS)
    logits, yt = predict_logits(model, tl)
    yp = logits.argmax(1)
    row = {'config': cfg, 'repeat': rep, 'fold': k, 'seed': s, 'n_train': len(tr), 'n_val': len(va),
           'n_test': len(te), 'val_macro_f1': round(vf1 * 100, 2),
           'test_macro_f1_present': round(macro_f1_present(yt, yp) * 100, 2),
           'test_acc': round(accuracy_score(yt, yp) * 100, 2), 'minutes': round((time.time() - t0) / 60, 2)}
    pd.DataFrame([row]).to_csv(res_csv, mode='a', header=not res_csv.exists(), index=False)
    pd.DataFrame({'config': cfg, 'repeat': rep, 'fold': k, 'seed': s, 'file': meta.file.values[te],
                  'cls': meta.cls.values[te], 'y_true': yt, 'y_pred': yp}).to_csv(
        pred_csv, mode='a', header=not pred_csv.exists(), index=False)
    left = (time.time() - t_start) / n * (len(todo) - n) / 3600
    print(f"[{n}/{len(todo)}] {cfg:<10} rep{rep} fold{k} seed{s}  F1 {row['test_macro_f1_present']:5.1f}"
          f"  ({row['minutes']:.1f} min, about {left:.1f} h left)", flush=True)
    del model
    torch.cuda.empty_cache()
print('All done')

## Run this cell in the morning and send the whole output to Claude

## 8. Analysis

In [ ]:
from scipy import stats

r = pd.read_csv(OUT_DIR / 'results.csv')
sz = r.groupby(['repeat', 'fold'])[['n_train', 'n_val', 'n_test']].first()
factor = 1 / len(sz) + sz.n_test.mean() / (sz.n_train + sz.n_val).mean()
pf = r.groupby(['config', 'repeat', 'fold']).test_macro_f1_present.mean().unstack('config')
print('Runs completed per config:', r.config.value_counts().to_dict())
print(f'Correction factor {factor:.3f}\n')
print('Mean macro F1 (%, mean ± SD over 40 runs)')
for c in CONFIGS:
    v = r[r.config == c].test_macro_f1_present
    if len(v):
        print(f'  {c:<10} {v.mean():6.2f} ± {v.std(ddof=1):5.2f}   (n={len(v)})')


def corrected(d):
    d = np.asarray(d, float)
    J = len(d)
    se = np.sqrt(factor * d.var(ddof=1))
    t = d.mean() / se
    h = stats.t.ppf(0.975, J - 1) * se
    return d.mean(), d.mean() - h, d.mean() + h, 2 * stats.t.sf(abs(t), J - 1), int((d > 0).sum()), J


rows = []
for c in PRIMARY:
    if c in pf and 'none_224' in pf:
        d = (pf[c] - pf['none_224']).dropna()
        if len(d) >= 3:
            rows.append((c, *corrected(d)))
# Holm
order = sorted(range(len(rows)), key=lambda i: rows[i][4])
m = len(rows)
adj = [None] * m
running = 0
for rank, i in enumerate(order):
    running = max(running, min(1, (m - rank) * rows[i][4]))
    adj[i] = running
print('\nPrimary comparisons (paired with none_224, 20 folds)')
print(f'  {"config":<10} {"diff":>7} {"95% CI":>18} {"p":>7} {"Holm p":>7} {"folds up":>8}  verdict')
for (c, mu, lo, hi, p, w, J), pa in zip(rows, adj):
    ok = (mu >= 2) and (pa < 0.05) and J == 20
    print(f'  {c:<10} {mu:+7.2f} [{lo:+6.2f}, {hi:+6.2f}] {p:7.3f} {pa:7.3f} {w:>4}/{J:<3}  {"✅ improved" if ok else "—"}')

if {'lfa14_320', 'none_320'} <= set(pf):
    d = (pf['lfa14_320'] - pf['none_320']).dropna()
    if len(d) >= 3:
        mu, lo, hi, p, w, J = corrected(d)
        print(f'\nSecondary: lfa14_320 vs none_320  {mu:+.2f} [{lo:+.2f}, {hi:+.2f}]  p={p:.3f}  {w}/{J}')

old = SRC_DIR / 'results.csv'
if old.exists() and 'none_224' in pf:
    o = pd.read_csv(old)
    o = o[o.config == 'none'].set_index(['repeat', 'fold', 'seed'])
    n2 = r[r.config == 'none_224'].set_index(['repeat', 'fold', 'seed'])
    j = n2.join(o[['test_acc']], rsuffix='_old', how='inner')
    if len(j):
        print(f'\nRerun agreement: mean |Δ| in accuracy between none_224 and round-1 none is {np.abs(j.test_acc - j.test_acc_old).mean():.2f} points'
              f'({len(j)} pairs, {(j.test_acc == j.test_acc_old).sum()} identical)')

sg = r.pivot_table(index=['config', 'repeat', 'fold'], columns='seed', values='test_macro_f1_present')
if sg.shape[1] == 2:
    print(f'\nSeed-only difference, mean |seed0 − seed1|: {(sg[0] - sg[1]).abs().mean():.2f} points')
print('\nMean time per run (minutes):', r.groupby('config').minutes.mean().round(2).to_dict())